# NB1 · Veriye ulaşmak

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Atölyede ne yapılıyor

Altı defter boyunca çalışan bir klinik karar destek sistemi yazılacaktır. Sistem bir
bilgisayar programıdır ve her defterde üzerine bir katman eklenir.

| Defter | Eklenen katman |
|---|---|
| NB1 | Veriye ulaşma ve ilk bakış |
| NB2 | Veriyi temizleme, bilgileri hazırlama, eğitim ve sınama grubuna ayırma |
| NB3 | Modelin kurulması, öğretilmesi ve başarımının ölçülmesi |
| NB4 | Modelin kararını gerekçelendiren katman |
| NB5 | Güvenlik bariyerleri ve uyumluluk raporu |
| NB6 | Web tarayıcısından kullanılan arayüz |

**Python bilmeniz gerekmiyor.** Kodu siz yazmayacaksınız. Her adımda size bir istem
verilir; bu istemi bir üretken yapay zekâ aracına (ChatGPT, Claude, Gemini) aktarır,
aracın verdiği kodu defterdeki boş hücreye yapıştırıp çalıştırırsınız.

Kodun ne yaptığını anlamanız ise beklenir. Her adımdan sonra kısa bir Python notu gelir.
Bu notlar, elinize gelen kodda gördüğünüz yapıları açıklar.


## Defterin kullanımı

Her istemin sonunda BEKLENEN SONUÇ başlıklı bir bölüm bulunur. Bu bölüm kodun ne
üretmesi gerektiğini yazar: Hangi adla hangi bilgi hazır olacak. Yapıştırma hücresinin
ardından gelen kontrol hücresi tam olarak bunu sınar.

Kontrol eksik bildirdiğinde kodu yapay zekâ aracına geri veriniz, kontrolün yazdığı
eksiği aynen aktarınız ve yeniden ürettiriniz. İlk denemede tutmaması olağandır.

Yapıştırma hücrelerinin ilk satırında `#@cdss adim_adi` yazar. **Bu satırı silmeyiniz.**
Kodunuzu onun altına yapıştırınız. Defterin sonunda bütün adımlar tek bir blok hâlinde
toplanır ve o bloğu bir sonraki deftere taşırsınız.


## Kurulum

Aşağıdaki hücre kontrol yardımcısını indirir. Atölyede size hazır verilen tek kod budur;
geri kalan her şeyi siz ürettireceksiniz.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır. Yapıştırma hücrelerindeki #@cdss satırını silmeyiniz.')


---

## Veri kümesi seçimi

Atölyede kullanabileceğiniz altı veri kümesi hazırlanmıştır. Hepsi açık erişimlidir, şifre
istemez ve Colab içinden doğrudan alınır. Birini seçiniz; sonraki bütün adımlar seçiminize
göre ilerleyecektir.

| Kod | Veri türü | Veri kümesi | Erişim | Tahmin edilecek durum |
|---|---|---|---|---|
| `mimic-yogun-bakim` | tablo | MIMIC-IV demo, 100 hastanın yoğun bakım kaydı | İnternetten doğrudan | Yatışın üç günü aşması |
| `wisconsin` | tablo | Breast Cancer Wisconsin, 569 örnek | scikit-learn içinde hazır | Kitlenin kötü huylu olması |
| `pnomoni-mnist` | görüntü | PneumoniaMNIST, 5.856 pediatrik akciğer grafisi | `pip install medmnist` | Pnömoni bulunması |
| `meme-mnist` | görüntü | BreastMNIST, 780 meme ultrasonu | `pip install medmnist` | Kitlenin kötü huylu olması |
| `mimic-ekg` | zaman serisi | MIMIC-IV-ECG demo, 92 hastadan 659 EKG | İnternetten doğrudan | Yatışın üç günü aşması |
| `sentetik-not` | metin | Üretilen Türkçe klinik notlar | Kod üretir | Kendi tanımladığınız durum |

### Seçerken bilmeniz gerekenler

**Hasta kimliği her kümede yok.** MIMIC yoğun bakım kaydında bir hastanın birden fazla
yatışı, EKG kümesinde bir hastanın birden fazla kaydı bulunur; NB2'deki hasta düzeyinde
ayrım dersi bu iki kümede gerçek karşılığını bulur. MedMNIST kümelerinde hasta kimliği yer
almaz, her görüntü ayrı bir kayıt sayılır. Wisconsin kümesinde de her satır ayrı bir
kişidir. Bu bir eksiklik değil, o kümelerin yapısıdır ve deftere not düşülür.

**EKG kümesi ile yoğun bakım kümesi aynı hastaları içerir.** EKG yolunu seçerseniz hedefi
klinik demodan alır, aynı durumu bu kez sinyalden tahmin etmeye çalışırsınız. İki yolu
karşılaştırma imkânı doğar.

**Metin yolunda gerçek veri yoktur.** Kimlik doğrulaması istemeyen Türkçe klinik not
kümesi bulunmamaktadır; MIMIC'in not modülü kimlik doğrulaması gerektirir. Bu yolda notlar
üretilir ve bu durum uyumluluk raporunda belirtilir.

**MedMNIST lisansı.** Kümeler CC BY 4.0 ile yayımlanmaktadır; DermaMNIST bunun dışındadır.
Buradaki iki küme CC BY 4.0 kapsamındadır.


---

## Adım 1 · Problemin tanımı ve programın başlangıcı

Her program birkaç hazırlık satırıyla başlar. İki şey yapılır: Kullanılacak hazır araç
takımları çağrılır ve program boyunca değişmeyecek değerler bir kez yazılır.

Bu değerler sizin probleminizi tarif eder. Sonraki bütün adımlar bunlara bakarak ilerler;
veri yükleme, temizleme ve arayüz istemlerinin hepsi buradaki tanımı okur. Bu yüzden
kendi probleminizle ilerliyorsanız değiştirmeniz gereken tek yer burasıdır.

**Ortak senaryo** şudur: Yoğun bakıma kabul edilen hastanın üç günden uzun kalıp
kalmayacağı, kabulden altı saat sonra öngörülecektir. Veri MIMIC-IV demo kümesinden
gelir.

**Kendi probleminizle** ilerleyecekseniz istemdeki beş metin alanını kendi problemenize
göre doldurunuz. Veri türünüz tablo değilse `VERI_TURU` değerini buna göre değiştiriniz;
sonraki adımlar bu değere bakarak dallanır.


### İstem 1

Aşağıdaki istemde köşeli parantez içindeki yerleri kendi problemine göre doldurunuz.
Ortak senaryoyla ilerliyorsanız hazır gelen değerleri aynen bırakınız.

```
Bir klinik karar destek sistemi yazmaya başlıyorum. Bu ilk hücre programın hazırlık
bölümü olacak.

Veri işleme ve makine öğrenmesi için gereken standart araç takımlarını çağır.

Sonra problemimi tarif eden değerleri tanımla. Her birinin yanına ne anlama geldiğini
Türkçe yaz:

  PROBLEM = 'Yoğun bakıma kabul edilen hastanın uzun süre kalıp kalmayacağının
             öngörülmesi'
  KARAR_ANI = 'Hastanın yoğun bakıma girişinden altı saat sonra'
  HEDEF_TANIMI = 'Yoğun bakım yatışının üç günü aşması'
  VERI_SETI = 'mimic-yogun-bakim'
  VERI_TURU = 'tablo'
  VERI_KOKU = 'https://physionet.org/files/mimic-iv-demo/2.2'
  RASTGELE_TOHUM = 42

VERI_SETI değeri şunlardan biri olabilir: 'mimic-yogun-bakim', 'wisconsin',
'pnomoni-mnist', 'meme-mnist', 'mimic-ekg', 'sentetik-not'.
VERI_TURU değeri şunlardan biri olabilir: 'tablo', 'goruntu', 'sinyal', 'metin'.
Veri kaynağım yoksa VERI_KOKU değerini 'sentetik' yaz.

VERI_TURU değeri 'tablo' ise ayrıca şu iki sayıyı da tanımla; bunlar yalnızca yoğun
bakım senaryosuna aittir:
  KARAR_PENCERESI_SAAT = 6      karar anına kadar geçen saat
  HEDEF_ESIK_GUN = 3            bu süreden uzun yatış hedef sayılır

Kullandığın araç takımlarının sürümlerini ekrana yaz. Ayrıca PROBLEM, KARAR_ANI,
HEDEF_TANIMI ve VERI_TURU değerlerini de yazdır ki ne üzerinde çalıştığım görünsün.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
Hücre çalıştıktan sonra şu adlar kullanılabilir olmalı:
  pd, np, PROBLEM, KARAR_ANI, HEDEF_TANIMI, VERI_SETI, VERI_TURU, VERI_KOKU,
  RASTGELE_TOHUM
```



In [ ]:
#@cdss hazirlik
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 1


In [ ]:
kit.check_defined('pd', 'np', 'PROBLEM', 'KARAR_ANI', 'HEDEF_TANIMI',
                  'VERI_SETI', 'VERI_TURU', 'VERI_KOKU', 'RASTGELE_TOHUM')


### Python notu · Gelen kodda ne var

Ürettiğiniz kodun en üstünde `import pandas as pd` gibi satırlar göreceksiniz.
**Kütüphane**, başkalarının yazdığı hazır bir araç takımıdır. `import` onu programa
çağırır. `as pd` ise ona kısa bir ad verir; bundan sonra `pandas` yerine `pd` yazmak
yeterlidir. `pandas` tablolarla çalışmak, `numpy` sayılarla hesap yapmak içindir.

`KARAR_PENCERESI_SAAT = 6` satırı bir **değişken** tanımlar. Büyük harfle yazılması, bu
değerin program boyunca değişmeyeceğini okuyan kişiye bildirir. Python bunu zorunlu
tutmaz, bir anlaşmadır.

Bu değerlerin en üstte durması önemlidir. Kodun ortasına gömülmüş bir `6` sayısı altı ay
sonra kimsenin anlamını hatırlamadığı bir değere dönüşür. Klinik bir sistemde eşik
değerleri denetime tabidir ve tek bir yerde görünmeleri gerekir.

`RASTGELE_TOHUM` şunu sağlar: Makine öğrenmesi adımlarının bir kısmı rastgelelik içerir.
Tohum sabitlenmediğinde program her çalıştırmada biraz farklı sonuç verir ve hangi
değişikliğin neye yol açtığı izlenemez.


In [ ]:
# Bu hücre hazır gelir. Tanımladığınız problemi ekrana yazar.
print('Problem    :', PROBLEM)
print('Karar anı  :', KARAR_ANI)
print('Hedef      :', HEDEF_TANIMI)
print('Veri kümesi:', VERI_SETI)
print('Veri türü  :', VERI_TURU)


---

## Adım 2 · Veriyi getirmek

Seçtiğiniz veri kümesine karşılık gelen istemi kullanınız. Altı istem de aynı yapıda bir
sonuç üretir: `hasta_id` ve `hedef` sütunlarını içeren bir tablo. Ortak yapı sayesinde
sonraki bütün adımlar veri türünden bağımsız olarak aynı şekilde ilerler.

İstemlerin içine dosya adreslerini, sütun adlarını ve paket adlarını ben yazdım. Yapay
zekâ aracının bunları kendiliğinden bilmesini beklemeyiniz; bilmediği bir şeyi tahminle
doldurur ve tahmini çoğunlukla yanlış olur. **Elinizdeki verinin neyi içerdiğini araca siz
söylersiniz.** Atölyenin en çok işinize yarayacak alışkanlığı budur.


### İstem 2a · `mimic-yogun-bakim` (tablo, ortak senaryo)

Adım 1'de tanımlanan `KARAR_PENCERESI_SAAT` ve `HEDEF_ESIK_GUN` sabitlerini kullanır.


```
Yoğun bakıma yatan hastaların verisini programa al. Veri internette açık erişimli ve
adresi yukarıda VERI_KOKU olarak tanımlı.

Üç dosya var, hepsi sıkıştırılmış tablo dosyası:
  {VERI_KOKU}/hosp/patients.csv.gz
      hastanın kimlik numarası (subject_id), cinsiyeti (gender), yaşı (anchor_age)
  {VERI_KOKU}/hosp/admissions.csv.gz
      hasta kimliği (subject_id), hastane yatış numarası (hadm_id),
      yatışın türü (admission_type), sigorta bilgisi (insurance)
  {VERI_KOKU}/icu/icustays.csv.gz
      hasta kimliği (subject_id), hastane yatış numarası (hadm_id),
      yoğun bakım yatış numarası (stay_id), hangi birim (first_careunit),
      giriş zamanı (intime), çıkış zamanı (outtime),
      yoğun bakımda kaç gün kaldığı (los)

Şunları yap:
1. Yoğun bakım yatışlarından başla. Karar anına ulaşmadan çıkan hastaları listeden
   çıkar; yani yoğun bakımda KARAR_PENCERESI_SAAT saatten az kalanları ele.
2. Tahmin edeceğimiz durumu oluştur: Yatış HEDEF_ESIK_GUN gününü aşıyorsa 1, aşmıyorsa
   0. Bunun adı hedef olsun.
3. Hasta kimliğinin adını hasta_id yap.
4. Diğer iki dosyadan hastanın cinsiyetini, yaşını, yatış türünü ve sigorta bilgisini
   getir. Bu işlem sırasında satır sayısı değişmemeli; değişirse uyar.
5. Bu madde çok önemli: Hastanın yoğun bakımda kaç gün kaldığı (los) ve ne zaman çıktığı
   (outtime) bilgilerini sonuçtan çıkar. Bu iki bilgi ancak hasta çıktıktan sonra
   öğrenilir, karar anında elimizde yoktur.

Bütün bunları ham_veri_yukle adında bir işlem parçasının içine koy. Başına ne yaptığını
anlatan bir açıklama yaz. Sonra onu çalıştır ve sonucu kohort adıyla sakla. Kohortun ilk
beş satırını göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
ham_veri_yukle adında çalıştırılabilir bir işlem parçası olmalı ve bir tablo döndürmeli.
kohort adında bir tablo hazır olmalı.
Bu tabloda hasta_id, stay_id, hedef, gender, anchor_age, admission_type, insurance
ve first_careunit sütunları bulunmalı.
Bu tabloda los ve outtime sütunları BULUNMAMALI.
```




### İstem 2b · `wisconsin` (tablo)

```
Breast Cancer Wisconsin veri kümesini programa al. Bu küme scikit-learn içinde hazır
gelir, internetten indirmeye gerek yoktur; load_breast_cancer ile açılır.

Şunları yap:
1. Kümeyi yükle ve bir tabloya çevir.
2. Hedefi hedef adıyla oluştur: Kitle kötü huyluysa 1, iyi huyluysa 0.
3. Her satır ayrı bir kişiye aittir; hasta_id sütununu satır sırasından üret ve bunun
   gerçek bir hasta kimliği olmadığını açıklama satırında belirt.

Bunları ham_veri_yukle adında bir işlem parçasının içine koy, başına ne yaptığını
anlatan bir açıklama yaz, çalıştır ve sonucu kohort adıyla sakla. İlk beş satırı göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
kohort adında bir tablo hazır olmalı; hasta_id ve hedef sütunlarını ve otuz ölçüm
sütununu içermeli.
```


### İstem 2c · `pnomoni-mnist` veya `meme-mnist` (görüntü)

```
MedMNIST koleksiyonundan bir görüntü kümesini programa al.

Kurulum: pip install medmnist
VERI_SETI değeri 'pnomoni-mnist' ise PneumoniaMNIST, 'meme-mnist' ise BreastMNIST
kullanılacak. Görüntüler 28x28 boyutunda ve tek kanallıdır. Küme eğitim, doğrulama ve
sınama olarak zaten bölünmüş gelir; üçünü birleştirip tek bir tablo yap, bölme işini
NB2'de kendimiz yapacağız.

Şunları yap:
1. Kümeyi indir ve yükle.
2. Her görüntüyü bir satır olacak biçimde tabloya koy.
3. Hedefi hedef adıyla oluştur; küme zaten ikili etiketlidir.
4. Bu kümede hasta kimliği yoktur. hasta_id sütununu satır sırasından üret ve bunun
   gerçek bir hasta kimliği olmadığını, dolayısıyla hasta düzeyinde ayrımın burada
   denetlenemeyeceğini açıklama satırında belirt.
5. Kümeyi çok büyük bulursan ilk 2000 görüntüyle sınırla ve bunu ekrana yaz.

Bunları ham_veri_yukle adında bir işlem parçasının içine koy, başına ne yaptığını
anlatan bir açıklama yaz, çalıştır ve sonucu kohort adıyla sakla. İlk beş satırı göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
kohort adında bir tablo hazır olmalı ve şu sütunları içermeli:
  hasta_id  -> satır sırasından üretilmiş kimlik
  goruntu   -> görüntünün kendisi
  hedef     -> 0 veya 1
```


### İstem 2d · `mimic-ekg` (zaman serisi)

```
MIMIC-IV-ECG demo kümesindeki EKG kayıtlarını programa al ve hedefi MIMIC-IV klinik demo
kümesinden getir. İki küme aynı 92 hastayı içerir.

Kurulum: pip install wfdb

EKG kayıtları şu adreste: https://physionet.org/files/mimic-iv-ecg-demo/0.1/
Kayıtlar WFDB biçimindedir, on saniye uzunluğunda, 500 Hz, on iki kanallıdır. Her hastanın
kayıtları kendi alt klasöründe durur ve klasör adı hastanın kimlik numarasıdır. Kayıt
listesi record_list.csv dosyasındadır.

Hedef için klinik demo kümesini kullan:
  https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz
  sütunlar: subject_id, stay_id, los (yoğun bakımda kaç gün kalındığı)

Şunları yap:
1. record_list.csv dosyasını oku ve ilk 200 kaydı al; hepsini indirmek uzun sürer.
2. Her kaydı wfdb ile oku ve yalnızca birinci kanalı (lead I) sakla.
3. Klinik demodan her hasta için en uzun yoğun bakım yatışını bul. Yatış üç günü
   aşıyorsa hedef 1, aşmıyorsa 0 olsun. Yoğun bakım kaydı bulunmayan hastaları at.
4. Hastanın kimliğini hasta_id adıyla sakla. Bir hastanın birden fazla EKG kaydı
   olabilir; bunu ekrana yaz.

Bunları ham_veri_yukle adında bir işlem parçasının içine koy, başına ne yaptığını
anlatan bir açıklama yaz, çalıştır ve sonucu kohort adıyla sakla. İlk beş satırı göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
kohort adında bir tablo hazır olmalı ve şu sütunları içermeli:
  hasta_id  -> hasta kimliği, aynı hastanın birden fazla satırı olabilir
  sinyal    -> EKG parçası
  hedef     -> 0 veya 1
```


### İstem 2e · `sentetik-not` (metin)

```
Klinik notlar üzerinden çalışan bir karar destek sistemi için veri hazırla.

Kimlik doğrulaması istemeyen Türkçe klinik not kümesi bulunmadığından notlar üretilecek.

Yukarıda tanımladığım değerleri kullan:
  PROBLEM        -> çözmek istediğim klinik problem
  HEDEF_TANIMI   -> tahmin edilecek durum
  RASTGELE_TOHUM -> tekrar üretilebilirlik için

Notlar Türkçe olsun ve gerçek notları zorlaştıran özellikleri taşısın: Kısaltmalar,
olumsuzlama, belirsizlik ifadeleri, her notta tekrarlanan şablon cümleler ve önceki nottan
kopyalanmış bölümler. Aradığımız durum tek bir kelimeden anlaşılmasın; hangi yüzeysel
ipuçlarını bilerek elediğini söyle. Aynı hastaya ait birden fazla not bulunsun.

Bunları ham_veri_yukle adında bir işlem parçasının içine koy, başına ne yaptığını
anlatan bir açıklama yaz, çalıştır ve sonucu kohort adıyla sakla. İlk beş satırı göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
kohort adında bir tablo hazır olmalı ve şu sütunları içermeli:
  hasta_id  -> hasta kimliği, aynı hastanın birden fazla satırı olabilir
  metin     -> klinik not
  hedef     -> 0 veya 1
```


In [ ]:
#@cdss veri_yukleme
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 2

İki kontrol var. Birincisi işlem parçasının çalışıp çalışmadığına, ikincisi ortaya çıkan
tablonun beklenen sonuca uyup uymadığına bakar. Ortak senaryo dışındaysanız ikinci
hücredeki sütun adlarını kendi verinize göre değiştiriniz.


In [ ]:
kit.check_function('ham_veri_yukle', call_with=((), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(
    kohort, name='kohort',
    required=['hasta_id', 'hedef'],
    forbidden=['los', 'outtime'],
    min_rows=30,
)


### Python notu · Tablo ve işlem parçası

Gelen kodda `pd.read_csv(...)` gibi satırlar ve `kohort` adlı bir nesne göreceksiniz.
Bu nesne bir **tablodur**; Python dünyasında adı *DataFrame*. Excel sayfası gibi
düşünebilirsiniz: Satırlar kayıtları, sütunlar bilgi alanlarını tutar. Bizim tablomuzda
her satır bir yoğun bakım yatışı, her sütun o yatışa ait bir bilgidir.

`def ham_veri_yukle():` satırı bir **fonksiyon** tanımlar. Fonksiyon, adı olan bir iş
parçasıdır. Bir kez tanımlanır, istendiği kadar çalıştırılır. İşi fonksiyona almanın üç
faydası vardır: İş tek bir yerde durur, tekrar çalıştırılabilir ve sınanabilir.

Fonksiyonun hemen altındaki üç tırnak içindeki metne **açıklama metni** denir. Yorum
satırından farkı, program çalışırken okunabilmesidir. Aşağıdaki hücre bunu gösterir.

`return` ifadesi fonksiyonun sonucu geri vermesini sağlar. Sonuç geri verilmezse
fonksiyonun içinde kalır ve kimse kullanamaz.


In [ ]:
# Fonksiyonun açıklama metnini okuyunuz.
help(ham_veri_yukle)


In [ ]:
# Tablonun boyutu ve ilk satırları.
print('Satır ve sütun sayısı:', kohort.shape)
print('Sütun adları:', list(kohort.columns))
kohort.head()


### Neden iki sütunu çıkardık

İstemin beşinci maddesinde `los` ve `outtime` sütunlarının çıkarılmasını istedik. Sebebi
şudur: Hastanın kaç gün kaldığı ancak çıktıktan sonra bilinir. Bu bilgi modele verilirse
model kusursuza yakın bir başarım gösterir, çünkü cevabı kendisine söylemiş oluruz.

Sistem hastanede devreye alındığında ise karar anında bu sütun boştur ve sistem hiçbir
işe yaramaz. Buna **sızıntı** denir. Üretken yapay zekâ ile yazılan klinik kodda en sık
rastlanan hatadır; hata mesajı vermez, sessizce ilerler ve sonucu iyileştirdiği için ilk
bakışta olumlu görünür.

Sızıntıyı sonradan yakalamaya çalışmak yerine istem aşamasında engelledik. Kontrol
hücresindeki `forbidden` listesi de bunu sınadı.


---

## Adım 3 · Veriye ilk bakış

Modele geçmeden önce elinizdeki tabloyu tanımanız gerekir. Bu adımda tabloyu özetleyen
bir işlem parçası yazdıracaksınız.


### İstem 3

```
kohort tablosunu özetleyen bir işlem parçası yaz. Adı veri_ozeti olsun ve kendisine
verilen tabloyu özetlesin.

Ekrana şunları yazsın:
  - kaç satır var,
  - kaç ayrı hasta var,
  - hedef durumun görülme oranı yüzde olarak.

Ayrıca her sütun için bir özet tablosu geri versin. Bu tabloda şunlar bulunsun:
  sutun       -> sütunun adı
  tip         -> içindeki bilginin türü
  eksik_oran  -> o sütunun ne kadarının boş olduğu
  benzersiz   -> kaç farklı değer içerdiği

Sonra bu işlem parçasını kohort üzerinde çalıştır, sonucu ozet adıyla sakla ve ozet
tablosunu göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
veri_ozeti adında çalıştırılabilir bir işlem parçası olmalı ve bir tablo döndürmeli.
ozet adında bir tablo hazır olmalı ve sutun, tip, eksik_oran, benzersiz sütunlarını
içermeli.
```


In [ ]:
#@cdss veri_kesfi
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 3


In [ ]:
kit.check_function('veri_ozeti', call_with=((kohort,), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(ozet, name='ozet',
                required=['sutun', 'tip', 'eksik_oran', 'benzersiz'])


### Python notu · Ekrana yazmak ile geri vermek

İstemde iki ayrı şey istedik: Bazı bilgilerin ekrana yazılması ve bir özet tablosunun
geri verilmesi. Aradaki fark önemlidir.

`print(...)` ekrana yazar. Yazdığı şey ekranda kalır, program onu bir daha kullanamaz.
Sınanamaz da; kontrol hücresi ekrandaki metni göremez.

`return ...` ise sonucu geri verir. Geri verilen bir tablo saklanabilir, sınanabilir ve
sonraki adımlarda kullanılabilir. Bu yüzden özet tablosunu geri verdirdik; NB5'te
uyumluluk raporuna ekleyeceksiniz.

Kural olarak şunu aklınızda tutunuz: Bir sonucu yalnızca insana göstermek istiyorsanız
ekrana yazdırınız; programın da kullanmasını istiyorsanız geri verdiriniz.


### Özetin okunması

Üç noktaya bakınız.

**Hasta sayısı satır sayısından küçük mü?** Küçükse bir hastanın birden fazla yatışı var
demektir. NB2'de veriyi ayırırken bu durumu hesaba katmanız gerekecek.

**Hedef durumun oranı nedir?** Bu sayı aradığımız durumun kohorttaki sıklığıdır. NB3'te
başarım yorumlanırken tek başına en belirleyici sayı bu olacak.

**Hangi sütunlar büyük ölçüde boş?** NB2'de eksik değerlerin nasıl doldurulacağına karar
vereceksiniz. Şimdilik yalnızca not ediniz.


---

## Defter sonu · Kodun toplanması

Aşağıdaki hücre bu defterde ürettiğiniz üç adımı tek bir blok hâlinde toplar. Çıkan
bloğun tamamını kopyalayınız; NB2'nin ilk hücresine yapıştıracaksınız.

Blok ayrıca `cdss_nb1.py` adıyla kaydedilir. Colab oturumu kapandığında bu dosya silinir,
bu nedenle bloğu kendi bilgisayarınızda bir metin dosyasına da kopyalayınız.


In [ ]:
kod = kit.export(save_as='cdss_nb1.py')


## Bu defterde ne yapıldı

Sistemin ilk katmanı yazıldı. Program artık hazırlığını yapıyor, veriyi internetten
alıyor ve elindeki tabloyu özetliyor.

Kod üretilirken üç alışkanlık edinildi. Verinin neyi içerdiği araca açıkça söylendi ve
tahmin etmesine izin verilmedi. Her istemin sonuna beklenen sonuç yazıldı, gelen kod buna
göre sınandı. Karar anında elde bulunmayan bilgiler daha veri alınırken çıkarıldı.

NB2'de bu kodun üzerine veri temizleme, bilgilerin hazırlanması ve eğitim ile sınama
grubuna ayırma eklenecektir.

---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelir ve Türkiye'deki bir yoğun bakım
popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
